# Incompressible Fluid Mixing as Constrained Optimal Transport

Numerical companion to the paper.  Everything here calls into `mixing.py`,
which must sit next to this notebook.

Two systems are solved, both on the flat torus $\Omega = [0,2\pi)^2$:

**Geodesic equations (Section 6.1)** --- the enstrophy-penalized case, where
$\rho$ and the costate $\lambda$ are transported together,

$$\partial_t \rho = -\nabla\rho \cdot v, \qquad
  \partial_t \lambda = -\nabla\lambda \cdot v, \qquad
  v = \Pi\big[\Delta^{-1}(\rho\,\nabla\lambda)\big].$$

**Gradient flow (Section 7.1)** --- steepest descent of the homogeneous
$\dot H^{-1}$ energy $E(\rho) = \tfrac12\|\rho - \rho^*\|^2_{\dot H^{-1}}$,

$$\partial_t \rho = -\nabla\cdot(\rho v), \qquad
  v = \Pi\big[\Delta^{-1}(\rho\,\nabla\,\delta E(\rho))\big], \qquad
  \delta E(\rho) = -\Delta^{-1}\tilde\rho .$$

The two share a velocity pipeline: the only difference is whether the scalar
potential is the transported costate or the first variation of the energy.
$\Pi$ is the Leray projection and $\Delta^{-1}$ the inverse Laplacian, both
applied spectrally.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import mixing as mx

%matplotlib inline

## Geodesic equations (Section 6.1)

In [ ]:
geodesic = mx.run_geodesic_equations(
    mx.Settings(
        N=128,
        final_time=mx.DEFAULT_FINAL_TIME["geodesic"],
        cfl=0.45,
        rho_expr="sin(x)*cos(y)",
        lambda_expr="cos(x)*sin(y)",
    )
)

print(mx.diagnostics_report(geodesic))

Every stage of the velocity pipeline at the final time.  Under
incompressibility the level sets of $\rho$ are only rearranged, never created
or destroyed, so the range of $\rho$ and all of its $L^p$ norms are conserved
by the exact dynamics --- the drift reported above is the numerical
dissipation of the first-order upwind scheme.

In [ ]:
mx.figure_overview(geodesic)

## Gradient flow (Section 7.1)

In [ ]:
gradient_flow = mx.run_gradient_flow(
    mx.Settings(
        N=128,
        final_time=mx.DEFAULT_FINAL_TIME["gradient-flow"],
        cfl=0.45,
    )
)

print(mx.diagnostics_report(gradient_flow))

In [ ]:
mx.figure_overview(gradient_flow)

The energy dissipation equality says $E$ can only decrease along the flow.
That is the sharpest check that the code is really implementing a gradient
flow of $E$, so the energy is recorded at every step rather than only at the
display snapshots.

In [ ]:
mx.figure_energy_history(gradient_flow)

In [ ]:
mx.figure_diagnostics(gradient_flow)

## Looking at a single snapshot

`data["snapshots"]` is a list of dictionaries, one per recorded frame, holding
the state, every stage of the velocity pipeline, and the diagnostics for that
instant.  The keys are listed in `mx.quantities_for(mode)`.

In [ ]:
table = mx.quantities_for(gradient_flow["mode"])
snapshot = gradient_flow["snapshots"][len(gradient_flow["snapshots"]) // 2]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
mx.FieldPanel(fig, axes[0], gradient_flow["grid"], snapshot, table["rho"], colorbar=True)
mx.FieldPanel(fig, axes[1], gradient_flow["grid"], snapshot, table["velocity v"], colorbar=True)
fig.suptitle(f"t = {snapshot['time']:.2f}")
fig.tight_layout()

## Writing the figures out

`save_run_figures` writes the standard set of PNGs used in the paper.  The same
thing is available from the command line:

```
python mixing.py geodesic      --grid 128
python mixing.py gradient-flow --grid 128
```

and `python mixing_gui.py` opens an interactive viewer where you can scrub
through the snapshots.

In [ ]:
for path in mx.save_run_figures(gradient_flow, "results"):
    print(path)